# Water Qualty model and indices
- This is a notebook of functions
- As Quang notes, the framework for water quality models is laid out in Ritchie et al. 2003 
--- Ritchie, C.J.; Zimba, V.P.; Everitt, H.J. Remote sensing techniques to assess water quality. Photogramm. Eng. Remote Sens. 2003, 69, 695–704. [Google Scholar] [CrossRef]
## 1. Chlorophyll indices and models 

## Chl-a 
### NDCI: 
(red_edge-red)/(red_edge + red). 
red edge generally considered as (670-760), Red 
For Sentinel-2 this can be any of B5,6,7 as the red edge. B4 as red. Other work uses B5 and B6. Interesting to see which gives the greatest contrast... Over a limited area and without any data correction, 6/4 and 7/4 give very similar results.. None of these ratios give credible results for 'raw' S2 geomedian data so far as I can see. The data are plauged by brdf issues.
For Landsat there is no red edge band. The superior calibration of Landsat may pay off..
In the past we have used B5/B4 with Landsat in australia for Lake Burley Griffin, and Shou et al in 2019 developed a model using bands 3 and 4 (Red and Green, 530-590)


follows from Mishra, S., & Mishra, D. R. (2012). Normalized difference chlorophyll index: A novel model for remote estimation of chlorophyll-a concentration in turbid productive waters. Remote Sensing of Environment, 117, 394-406.

Landsat equivalent would be 


### Chl-a models:
Guided by Zhou et al, global study of lakes, who are not sharing data or full details of their algorithms, the approach would be to:

#### Multi-algorithm depending on water body type:

Apply a Chla algorithm suited to the water body type; there are 3 possibilities: 

a) NASA OCx for clear waters. This model is suited to <.2 mg/m3 of Chl-a, which is for most purposes 'clean water'. This will be of marginal relevance to DE Africa, and can be put aside for now.

b) Two-band ratio of Gurlin et al for eutrophic and blue to green waters. This provides the best estimates in turbid productive waters.  It is base on NIR and IR, and calibrated to MERIS / MODIS. 

### Specifics:
#### MERIS

25.28 x MERIS_2BM^2 + 14:85 x MERIS_2BM – 15.18 

where MERIS_2BM = band9(703.75–713.75 nm) / band7(660–670 nm)

closest S2 bands are : B5 (703.9) and B4(664) respectively
closest OLI bands are: .... Band 5 (850-880) and, B4 (640-670), the red band - not good enough?
closest TM  bands are: B4 (760-790) and, B3 (630-690), -- not good enough?

#### MODIS
Chla = 190.34 x MODIS_2BM – 32.45
where: MODIS_2BM = Band15(743–753 nm) / Band13(662–672 nm)

closest S2 bands are : B6(740) and B4(664)
closest OLI bands are : B5 (850-880) and B4 (640-670)  - not close enough?
closest TM bands are : B4  (760-790) and, B3 (630-690) - worth a try

TM looks like a better fit than OLI in both cases. For OLI stick to the ndci.

c) TC2 of Lui et al for hypereutrophic and green-to-brown waters. These waters are highly relevant in cases where there are, for example, high sediment loads in rivers and high nutrient inputs to waterbodies. 




#### Normalised Difference Chlorophyll Index - NDCI
- Band ratios, NIR v red
- MSI has three NIR bands so the index can be validly calculated in 3 ways
- OLI bands 5 and 4 are NIR/Red
- TM  bands 4 and 3 apply

In [9]:
def NDCI_NIR_R(dataset, NIR_band, red_band):
# ---- Function to calculate ndci from input nir and red bands 
    if verbose: print("NDCI_NIR_R")
    return  (dataset[NIR_band] - dataset[red_band])/(dataset[NIR_band] + dataset[red_band])
   

#### Chl-a models - two band model of Gurlin et al
- These models assume a surface reflectance corrected by 2pi and possibly other pre-processing steps
- The quality of results needs to be monitord - as with all.

In [7]:
'''
MERIS two band:
    meris2b = 25.28 x MERIS_2BM^2 + 14:85 x / MERIS_2BM – 15.18 
    where MERIS_2BM = band9(703.75–713.75 nm) / band7(660–670 nm)
    closest S2 bands are : B5 (703.9) and B4(664) respectively
    may need to divide by pi 

     MERIS2B = 25.28 * ((ds.msi05 / ds.msi04r)*2) + 14.85 * (ds.msi05 / ds.msi04r) - 15.18

     MODIS
     Chla = 190.34 x MODIS_2BM – 32.45
     where: MODIS_2BM = Band15(743–753 nm) / Band13(662–672 nm)
     closest S2 bands are : B6(740) and B4(664)

     MODIS2B = 190.34 * (ds.msi06/ ds.msi04r) - 32.45
'''

# Function to estimate ChlA using the MERIS and MODIS 2-Band models, using closest bands from MSI (6, 5, 4) 
def ChlA_MERIS2B(dataset, band_708, band_665):   #matching MSI bands are 5 and 4
    if verbose: print("ChlA_MERIS two-band model")
    X = dataset[band_708] / dataset[band_665]
    return  (25.28 * (X)*2) + 14.85 * (X) - 15.18


def ChlA_MODIS2B(dataset, band_748, band_667):  #matching MSI bands are 6 and 4
    if verbose: print("ChlA_MODIS two-band model")
    X = dataset[band_748] / dataset[band_667]
    return (190.34 * X) - 32.45
  

## 2. TSS indices and models

Models for turbidity measures from EO generally take the form of: 

- Y = A + B^X    
- Y = AB^X       
- Y = A + B ln X 
- ln Y = A + B^X

Where 
- X "could be reflectance, radiance, energy in a single band, or ratio of two bands."
- Y is the dependent variable
- A and B are coefficients determined by comparison with field data.
                    

## 2. TSS indices and models
Suspended sediment indices, generically, rely on band ratios of red and green.
- Multiple models are demonstrated in the notebook exmaple water_quality_suspended_matter
We should use these as an ensemble.

#### Normalised Difference Suspeneded Sediment Index NDSSI
- These are essentially  
  - red-green / red_+green, or,
  - blue-NIR / blue + NIR




In [5]:
def NDSSI_RG(dataset, red_band, green_band):
    if verbose: print("NDSSI_RG")
    return ((dataset[red_band] - dataset[green_band]) / (dataset[red_band] + dataset[green_band]))
def NDSSI_BNIR(dataset, blue_band, NIR_band):
    if verbose: print("NDSSI_BNIR")
    return ((dataset[blue_band] - dataset[NIR_band]) / (dataset[blue_band] + dataset[NIR_band]))

#### Turbidity index of Yu, X. et al. 

- **An empirical algorithm to seamlessly retrieve the concentration
of suspended particulate matter from water color across ocean to turbid river
mouths. Remote Sens. Environ. 235, 111491 (2019).**
- Used in screening turbid waters for mapping floating algal blooms
- Initially developed with TM
- -TI = ((Red − green) − (NIR − Rgreen)) ^ 0.5

In [8]:
def TI_yu(dataset,NIR,Red,Green,scalefactor=0.01,verbose=False):
    if verbose: print('TI_yu')
    return  scalefactor * ((dataset[Red] - dataset[Green]) - ((dataset[NIR] - dataset[Green]) * 0.5))


#### Lymburner Total Suspended Matter (TSM)
Paper: [Lymburner et al. 2016](https://www.sciencedirect.com/science/article/abs/pii/S0034425716301560)


Units of mg/L concentration

TSM for ETM:
$$
tss_etm_lym = 3983 \cdot (\frac{(\text{Green} + \text{Red}) * 0.0001}{2})^{1.6246}
$$

TSM for OLI:
$$
tss_oli_lym = 3957 \cdot (\frac{(\text{Green} + \text{Red}) * 0.0001}{2})^{1.6436}
$$



In [3]:
'''
these models developed by leo lymburner and arnold dekker are simple, stable, and produce credible results over a range of observations
'''
# Function to calculate Lymburner TSM value for ETM (Landsat 7)
def TSM_LYM_ETM(dataset, green_band, red_band, scale_factor=0.0001):
    if verbose: print("TSM_LYM_ETM")
    return 3983 * ((dataset[green_band] + dataset[red_band]) * scale_factor / 2) ** 1.6246

# Function to calculate Lymburner TSM value for OLI (Landsat 8)
def TSM_LYM_OLI(dataset, green_band, red_band, scale_factor=0.0001):
        if verbose: print("TSM_LYM_OLI")
        return 3957 * ((dataset[green_band] + dataset[red_band]) * scale_factor / 2) ** 1.6436



#### Suspended Particulate Model (SPM)
Paper: [Zhongfeng Qiu et.al. 2013](https://www.researchgate.net/publication/260152686_A_simple_optical_model_to_estimate_suspended_particulate_matter_in_Yellow_River_Estuary)

This paper does not give the model below, and its not 'et al'. The work of Qiu is limited to the Yellow River Estuary. This is a high dimenional model fitted to a local dataset, but surprisingly seems to work relatively well. The use of 'log' is also ambiguous; in the paper they talk of 'exponential', but that has been interpreted here as the log base 10. log(1) in numpy is e. 

{this Qiu paper also gives Zhang as: 

log[SPM] = 0.5937 + 19.02X1-0.6208X2	X1 = [Rrs(555) + Rrs(645)], X2 = [ Rrs(488)/Rrs(555)]	76.49	38.77	0.55}

Units of g/m^3 concentration

SPM for Landsat 8:
$$
X = 2.26*(\frac{\text{Red}}{\text{Green}})^3 - 
           5.42*(\frac{\text{Red}}{\text{Green}})^2 +
           5.58*(\frac{\text{Red}}{\text{Green}}) - 0.72
$$

$$
SPM = 10^{X} - 1.43 
$$

This model seems to discriminate well although the scaling is questionable and it goes below zero due to the final subtraction. The final subtraction seems immaterial in  and could be skipped. 

In [4]:
'''
Function to calculate Suspended Particulate Model value
This model seems to discriminate well although the scaling is questionable and it goes below zero due to the final subtraction.
The final subtraction seems immaterial and could be skipped. 
'''

def SPM_QIU(dataset,green_band,red_band):
    if verbose: print("SPM_QIU")
    return (
        10. ** (
              2.26 * ((dataset[red_band] / dataset[green_band]) ** 3) -
              5.42 * ((dataset[red_band] / dataset[green_band]) ** 2) +
              5.58 * ((dataset[red_band] / dataset[green_band])) -
              0.72
             )
         #- 1.43
         )

#### Quang Total Suspended Solids (TSS)
Paper: [Quang et al. 2017](https://www.mdpi.com/2073-4441/9/8/570)

Units of mg/L concentration

$$
Quang8 = 380.32 * \text{Red}*0.0001 - 1.7826
$$

In [19]:
''' 
This is a linear transformation of the 'red' band of the Oli sensor on Landsat-8
An empirical model developed for a small bay in vietnam, 
it  is unlikely to perform well over a wide range of situations. 
'''
# Function to calculate quang8 value
def TSS_QUANG8(dataset,red_band):
    if verbose: print("TSS_QUANG8")
    return 380.32 * (dataset.red_band) * 0.0001 - 1.7826

#### Model of Zhang et al 2023:
$$
LandsatTSS = 1.20 exp(14.44 X1) ; X1 = (G + R)/(B/R) for landsat.
$$

- Green = B3 520-600; Blue = B2 450-515 Red = B4 630-680
- Note that they also used an established linear cross-corellation method to link ETM to OLI. Results from MSI seem to better match the lymburner model..

The same relationship type should broadly hold for the respective bands of Sentinel-2.  
- Blue B2 496; Green = B3 560; Red = B4 664
- However this does not apper to produce reasonable results at this time.
- Dark pixel correction is making things worse at this point, for this model

In [ ]:
'''
function to calculate Zhang TSS 
This model seems to be funamentally unstable, using band ratios as an exponent is asking for extreme values and numerical overflows.
The function is not scale-less.
The factor of 0.0001 is not part of the forumula but  scales back from 10000 range which is clearly 
ridiculous (exp(10000) etc. is not a good number). This therefore avoids overflow. 
# base 10 or base e? The difference will be a constant shift in the results, so better to use e
# this model is running to infinity all over the place. 
# A better distribution is achieved with not applying the exponent..
This model can only be used together with other models and indices; it may handle some situations well...
'''
def TSS_Zhang(dataset, blue_band, green_band, red_band, scale_factor=0.0001,verbose=True):
    if verbose: print("TSS_Zhang")
    abovezero = .00001  #avoids div by zero if blue is zero
    GplusR = dataset[green_band] + dataset[red_band]             
    RdivB  = dataset[red_band] / (dataset[blue_band] + abovezero)   
    X = (GplusR * RdivB) * scale_factor                          
    #return(10**(14.44*X))*1.20
    #return  np.exp(14.44* X)*1.20
    return  (14.44* X)      #the distribution of results is exponential; this measure will be more stable without raising to the power.
      
